Additional Features:
 - Excess style momentum
 - Excess style volatility (against each other)
 - correlation between factor returns (against benchmark)
 - Excess style skewness (against benchmark)
 - Relative momentum (factors relative to each other, trend scan - categorical features)
 


In [14]:
# run script from the data_input file
import os
os.chdir('C:/Users/p528552/OneDrive - South African Reserve Bank/Documents/1. MEng - Data Science/1. Project_2025/Data/factor_timing/data_input')

In [15]:
# Import libraries:
import numpy as np
import pandas as pd
#from scipy.stats import kurtosis, skew      


In [16]:
# Import data:
df_factors = pd.read_csv("factor_returns.csv", parse_dates=['Date'])
df_factors = df_factors.set_index('Date')

#df_vol = pd.read_csv("local_global_vol.csv", parse_dates=['Date'])
#df_vol = df_vol.set_index('Date')

#df = pd.concat([df_factors,df_vol], axis=1, join='inner')
df = df_factors.dropna()

print(df)
#print(df.dtypes)


            Momentum     Value   Quality
Date                                    
2007-03-05  0.018613  0.015753 -0.010966
2007-03-12 -0.012887 -0.020257  0.048512
2007-03-19  0.060268  0.052751  0.014917
2007-03-26  0.022670  0.007935  0.007713
2007-04-02  0.010079  0.009888  0.018702
...              ...       ...       ...
2025-03-03  0.049395  0.011621  0.076581
2025-03-10 -0.009996 -0.002634  0.041062
2025-03-17  0.040341  0.006385  0.018397
2025-03-24  0.008068 -0.006651  0.020056
2025-03-31 -0.017642 -0.096445 -0.010625

[944 rows x 3 columns]


In [17]:
import itertools as it

import itertools as it

def factor_relative_core_features(df, window=12, window_1m=4, window_3m=12):

    feat = pd.DataFrame(index=df.index)
    factors = df.columns

    for a, b in it.combinations(factors, 2):
        name = f"{a.lower()}_vs_{b.lower()}"
        spread = df[a] - df[b]

        """
        # -------------------------
        # 1. Cumulative relative performance
        # -------------------------
        cum_a = (1 + df[a]).cumprod()
        cum_b = (1 + df[b]).cumprod()
        feat[f'cumrel_{name}'] = cum_a / cum_b - 1
        """
        # -------------------------
        # 2. Relative z-score (12-week default)
        # -------------------------
        roll_mean = spread.rolling(window).mean()
        roll_std  = spread.rolling(window).std()
        feat[f'zrel_{name}_{window}'] = (spread - roll_mean) / roll_std

        # -------------------------
        # 3. Rolling relative volatility
        # -------------------------
        feat[f'relvol_{name}_{window}'] = spread.rolling(window).std()

        # -------------------------
        # 4. Correlation between factors
        # -------------------------
        feat[f'corr_{a.lower()}_{b.lower()}_{window}'] = df[a].rolling(window).corr(df[b])

        # -------------------------
        # 5. Volatility ratio
        # -------------------------
        vol_a = df[a].rolling(window).std()
        vol_b = df[b].rolling(window).std()
        feat[f'volratio_{a.lower()}_{b.lower()}_{window}'] = vol_a / vol_b

        # -------------------------
        # 6. 1-Month Momentum (4 weeks)
        # -------------------------
        mom1_a = (1 + df[a]).rolling(window_1m).apply(np.prod, raw=True) - 1
        mom1_b = (1 + df[b]).rolling(window_1m).apply(np.prod, raw=True) - 1
        feat[f'mom1m_{name}'] = mom1_a - mom1_b

        # -------------------------
        # 7. 3-Month Momentum (12 weeks)
        # -------------------------
        mom3_a = (1 + df[a]).rolling(window_3m).apply(np.prod, raw=True) - 1
        mom3_b = (1 + df[b]).rolling(window_3m).apply(np.prod, raw=True) - 1
        feat[f'mom3m_{name}'] = mom3_a - mom3_b

    return feat



In [20]:
factor_relative_features = factor_relative_core_features(df, window=12)
factor_relative_features = factor_relative_features.dropna()
print(factor_relative_features)

            zrel_momentum_vs_value_12  relvol_momentum_vs_value_12  \
Date                                                                 
2007-05-21                   0.903263                     0.007903   
2007-05-28                   1.507127                     0.008974   
2007-06-04                   0.522944                     0.009001   
2007-06-11                  -0.835358                     0.009225   
2007-06-18                   0.637502                     0.008506   
...                               ...                          ...   
2025-03-03                   2.149018                     0.015960   
2025-03-10                  -0.614738                     0.016265   
2025-03-17                   1.538872                     0.017896   
2025-03-24                   0.366983                     0.017405   
2025-03-31                   2.433776                     0.026013   

            corr_momentum_value_12  volratio_momentum_value_12  \
Date                   

In [21]:
# Save output into csv file:
factor_relative_features.to_csv('factor_relative_features.csv', index=True)